In [37]:
import os
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# constants
SR = 16000
N_MELS = 128
FIXED_LEN = 128
BATCH_SIZE = 32
EPOCHS = 10

In [38]:
base_path = r"C:/Users/Amol/OneDrive/Desktop/New folder/ESC-50-master/ESC-50-master/audio"
meta_path = r"C:/Users/Amol/OneDrive/Desktop/New folder/ESC-50-master/ESC-50-master/meta/esc50.csv"

In [39]:
meta = pd.read_csv(meta_path)

print(meta.head())
print("Total samples:", len(meta))

            filename  fold  target        category  esc10  src_file take
0   1-100032-A-0.wav     1       0             dog   True    100032    A
1  1-100038-A-14.wav     1      14  chirping_birds  False    100038    A
2  1-100210-A-36.wav     1      36  vacuum_cleaner  False    100210    A
3  1-100210-B-36.wav     1      36  vacuum_cleaner  False    100210    B
4  1-101296-A-19.wav     1      19    thunderstorm  False    101296    A
Total samples: 2000


In [40]:
distress_classes = [
    "crying_baby",
    "glass_breaking",
    "sneezing",
    "dog",
    "rooster"
]

meta['label'] = meta['category'].apply(
    lambda x: 1 if x in distress_classes else 0
)

print(meta['label'].value_counts())

label
0    1800
1     200
Name: count, dtype: int64


In [41]:
def extract_features(file_path):
    try:
        audio, sr = librosa.load(file_path, sr=SR)

        mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS)
        mel_db = librosa.power_to_db(mel)

        # fix size
        if mel_db.shape[1] < FIXED_LEN:
            mel_db = np.pad(mel_db, ((0,0),(0,FIXED_LEN - mel_db.shape[1])))
        else:
            mel_db = mel_db[:, :FIXED_LEN]

        return mel_db

    except:
        return np.zeros((N_MELS, FIXED_LEN))

In [42]:
class AudioDataset(Dataset):
    def __init__(self, meta, base_path):
        self.meta = meta.reset_index(drop=True)
        self.base_path = base_path

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        row = self.meta.iloc[idx]
        file_path = os.path.join(self.base_path, row['filename'])

        features = extract_features(file_path)
        features = np.expand_dims(features, axis=0)

        label = row['label']

        return torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)

In [43]:
train_meta, test_meta = train_test_split(
    meta,
    test_size=0.2,
    random_state=42,
    stratify=meta['label']
)

In [44]:
train_dataset = AudioDataset(train_meta, base_path)
test_dataset = AudioDataset(test_meta, base_path)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [45]:
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.reshape(x.size(0), -1)
        return self.fc(x)

In [46]:
model = CNNModel().to(DEVICE)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [47]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE).unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")

Epoch 1/10, Loss: 23.6729
Epoch 2/10, Loss: 12.9778
Epoch 3/10, Loss: 11.4062
Epoch 4/10, Loss: 9.5661
Epoch 5/10, Loss: 7.4849
Epoch 6/10, Loss: 7.6734
Epoch 7/10, Loss: 5.8698
Epoch 8/10, Loss: 4.4270
Epoch 9/10, Loss: 4.1799
Epoch 10/10, Loss: 2.5293


In [48]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(DEVICE)

        outputs = model(X_batch)
        preds = (outputs > 0.5).cpu().numpy()

        all_preds.extend(preds.flatten())
        all_labels.extend(y_batch.numpy())

print(classification_report(all_labels, all_preds))

              precision    recall  f1-score   support

         0.0       0.96      0.96      0.96       360
         1.0       0.65      0.65      0.65        40

    accuracy                           0.93       400
   macro avg       0.81      0.81      0.81       400
weighted avg       0.93      0.93      0.93       400



In [49]:
torch.save(model.state_dict(), "cnn_baseline.pth")
print("Model saved ")

Model saved 


In [50]:
def predict_file(file_path):
    features = extract_features(file_path)

    features = np.expand_dims(features, axis=(0,1))
    features = torch.tensor(features, dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        prob = model(features).item()

    return prob, 1 if prob > 0.5 else 0


# example
test_file = os.path.join(base_path, meta.iloc[0]['filename'])

print(predict_file(test_file))

(0.8934048414230347, 1)
